In [0]:
# Import necessary libraries for Spark DataFrame operations and date handling
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime


In [0]:
# Define source path for raw CSV data and target Delta table for bronze layer
Source_Path = "s3://goodcabs-data-dp/data-store/city"
Target_Table = "transportation.bronze.city"


In [0]:
# Step 1: Read raw CSV data using Spark's read.csv method
# This reads the data with header and infers schema automatically
# Note: This cell is a simpler version; the next cell uses more robust options

# Read CSV file
df = spark.read.csv(Source_Path, header=True, inferSchema=True)

# Display sample rows to verify data ingestion
df.show()


In [0]:
# Step 1: Read raw CSV data with robust options for schema merging and corrupt record handling
# - 'mergeSchema': Allows schema evolution
# - 'columnNameOfCorruptRecord': Captures corrupt records
# - 'mode': 'PERMISSIVE' allows partial reads
# - '_metadata.file_path' is available for tracking source files

df = (
    spark.read
        .format("csv")\
        .option("header", "true")\
        .option("inferSchema", "true")\
        .option("mode", "PERMISSIVE")
        .option("mergeSchema", "true")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .load(Source_Path)
    )


In [0]:
# Step 2: Add metadata columns to track ingestion time and source file
# - 'ingesttime': Timestamp when data is ingested
# - 'filename': Source file path for traceability

df = (
    df.withColumn("ingesttime", current_timestamp())
      .withColumn("filename", col("_metadata.file_path"))
)


In [0]:
# Display the DataFrame with added metadata columns
# This helps verify that 'ingesttime' and 'filename' are correctly populated

df.show()


In [0]:
# Step 3: Write the DataFrame to a Delta table in the bronze layer
# - 'append' mode: Adds new data without overwriting existing
# - 'checkpointLocation': Enables streaming checkpointing (for incremental loads)
# - 'mergeSchema' and 'overwriteSchema': Support schema evolution
# - 'saveAsTable': Registers the table in the metastore

(df.write\
    .format("delta")\
    .mode("append")\
    .option("checkpointLocation", f"{Target_Table}/_checkpoints")
    .option("mergeSchema", "true")
    .option("overwriteSchema", "true")
    .saveAsTable(Target_Table)
)


In [0]:
# Query the bronze Delta table to verify data was written successfully
# Displays all rows from the table

spark.sql("select * from transportation.bronze.city").show()


In [0]:
# Step 4: Set essential table properties for the bronze Delta table
# - 'quality', 'layer', 'source_format': Business metadata
# - 'delta.autoOptimize.optimizeWrite', 'delta.autoOptimize.autoCompact': Performance optimizations
# - 'delta.enableChangeDataFeed': Enables CDC for downstream consumption
# Uncomment 'delta.autoOptimize.maxFileSize' if needed for file size tuning

spark.sql(f"""
ALTER TABLE {Target_Table} SET TBLPROPERTIES (
    quality = 'bronze',
    layer = 'bronze',
    source_format = 'csv',
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.enableChangeDataFeed' = 'true'
    -- 'delta.autoOptimize.maxFileSize' = '134217728'
)
""")
